## LLMをプログラムで操作する

既に多くの方は、ChatGPTのような大規模言語モデル（LLM）とやり取りしたことがあるかもしれません。通常、これはUIやアプリケーションを介して行われます。

このNotebookでは、Pythonを使用してLLMに直接アクセスします。
モデルとして、OpenShift AIでServingされた[**tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3**](https://huggingface.co/tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3)を利用します。
これはLlama 3.1の英語の能力を維持しながら日本語の能力を強化したオープンな大規模言語モデルで、米Meta社のLlama 3.1をベースに、東京科学大学情報理工学院の岡崎研究室と横田研究室、国立研究開発法人産業技術総合研究所の研究チームによって開発されたものです。

このモデルは既にラボクラスターにデプロイされています。小型モデルとはいえ、動作には24GBのRAMを持つGPUが必要です。

### 必要なライブラリとインポート

Labの指示に従って適切なワークベンチイメージを選択して起動した場合、必要なすべてのライブラリがすでにインストールされているはずです。もしインストールされていない場合は、次のセルの最初の行のコメントを外して正しいパッケージをすべてインストールしてください。その後、必要なライブラリをインポートします。

In [ ]:
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt # 正しいワークベンチイメージを選択していない場合のみ、コメントを外してください

import json
import os

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI

### Langchain

[Langchain](https://www.langchain.com/)は、言語モデルを活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLM（Language Model）に適切にクエリを発行するために手動で書かなければならないすべてのコードを処理します。

まず、LLMインスタンスを作成します。これはLLM APIへのクエリが行われる場所と、モデルに適用されるいくつかのパラメータによって定義されます。たとえば、`max_new_tokens`はモデルが最大512トークン（単語または単語の一部）で回答するよう指示します。`temperature`はここで非常に低く設定されており、モデルに真実に基づいたままであり、あまり「創造的」にならないように指示します。

In [ ]:
# LLM推論APIのURL
inference_server_url = "http://llama-3-1-swallow-8b-instruct-v0-3-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLMの定義
llm = VLLMOpenAI(
    openai_api_key="EMPTY", # OpenAI互換のAPIクライアントを使用していますが、モデルはOpenAIではなくOpenShift上で実行されています。そのため、api keyを指定しますが、これは使われません。
    openai_api_base= f"{inference_server_url}/v1",
    model_name="llama-3-1-swallow-8b-instruct-v0-3",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)


また、モデルに送信するすべてのリクエストに適用する**テンプレート**（プロンプト）も必要です。

モデルにクエリを送信する際、ほとんどの場合、ユーザーが入力した内容をそのまま送ることは望まれません。モデルがそれをどのように扱うかを正確に指示する必要があります。例えば、何をどのように回答するか、回答してはいけないこと、使用するべきトーンなどです。

In [ ]:
template="""<|begin_of_text|><|start_header_id|>system<|end_header_id|>


あなたは、親切で、礼儀正しく、正直なアシスタントです。
常に気配りと尊重をもって接し、真摯にサポートします。できる限り有用な返答を提供しますが、安全を確保します。
有害で、倫理に反する、偏見のある、または否定的な内容は避けます。返答が公正でポジティブなものであることを確認します。<|eot_id|><|start_header_id|>user<|end_header_id|>

与えられた質問に対して、できるだけ多くの情報を含めて回答して下さい。

### 質問:
{input}

### 回答:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

prompt = PromptTemplate(input_variables=["input"], template=template)

Langchainはこれらの要素を簡単に「つなぎ合わせ」、モデルにクエリを送信するために使用する**会話**オブジェクト (conversation) を作成することができます。

In [ ]:
conversation = prompt | llm

これでモデルにクエリを送信する準備が整いました！

In [ ]:
query = "人工知能（AI）とは、どのようなものですか？300文字程度で回答して下さい。"

conversation.invoke(input=query); # 行末の ";" は、最終の出力（ストリームされた回答の繰り返し）を非表示にします。